# Module 10 - 실습 1 솔루션: 토큰 예산 관리 (TokenBudget)

In [ ]:
import sys
sys.path.insert(0, '../..')

import tiktoken
print("tiktoken 준비 완료")

## 실습 1 솔루션: tiktoken으로 토큰 수 계산

In [ ]:
# 인코더 생성
enc = tiktoken.get_encoding("cl100k_base")

texts = {
    "영어": "Hello, world!",
    "한글": "안녕하세요, 세상!",
    "코드": "def fibonacci(n):\n    if n <= 1:\n        return n\n    return fibonacci(n-1) + fibonacci(n-2)",
}

for label, text in texts.items():
    tokens = enc.encode(text)
    char_count = len(text)
    token_count = len(tokens)
    print(f"{label}: {char_count}자 → {token_count}토큰 (비율: {token_count/char_count:.2f})")

In [ ]:
# 검증
english_tokens = len(enc.encode("Hello, world!"))
korean_tokens = len(enc.encode("안녕하세요, 세상!"))
assert english_tokens < len("Hello, world!")
assert korean_tokens > len("안녕하세요, 세상!") // 2
print(f"영어: {english_tokens}토큰, 한글: {korean_tokens}토큰")
print("✅ 실습 1 완료! 토큰 수 계산이 올바르게 동작합니다.")

## 실습 2 솔루션: TokenBudget 클래스

In [ ]:
class TokenBudget:
    """토큰 예산 관리.
    
    LLM에 보낼 프롬프트의 토큰 수를 정확하게 관리합니다.
    """
    
    def __init__(self, model: str = "cl100k_base", max_tokens: int = 8000):
        self._enc = tiktoken.get_encoding(model)
        self.max_tokens = max_tokens
        self._used = 0
    
    def count(self, text: str) -> int:
        """텍스트의 토큰 수를 반환합니다."""
        return len(self._enc.encode(text))
    
    @property
    def remaining(self) -> int:
        """남은 토큰 예산."""
        return max(0, self.max_tokens - self._used)
    
    @property
    def used(self) -> int:
        """사용한 토큰 수."""
        return self._used
    
    def consume(self, text: str) -> str | None:
        """예산 내에서 텍스트를 소비합니다."""
        tokens = self.count(text)
        if tokens <= self.remaining:
            self._used += tokens
            return text
        return None
    
    def truncate_to_budget(self, text: str, reserve: int = 100) -> str:
        """남은 예산에 맞게 텍스트를 토큰 단위로 절삭합니다."""
        available = self.remaining - reserve
        if available <= 0:
            return ""
        
        tokens = self._enc.encode(text)
        if len(tokens) <= available:
            self._used += len(tokens)
            return text
        
        # 토큰 단위로 절삭
        truncated = self._enc.decode(tokens[:available])
        self._used += available
        return truncated + "\n... [토큰 한도로 절삭됨]"

In [ ]:
# 검증
budget = TokenBudget(max_tokens=1000)
assert budget.remaining == 1000
assert budget.used == 0

text = "안녕하세요, 이것은 테스트입니다."
token_count = budget.count(text)
assert token_count > 0

result = budget.consume(text)
assert result == text
assert budget.used == token_count
assert budget.remaining == 1000 - token_count

print(f"토큰 수: {token_count}, 사용: {budget.used}, 남은: {budget.remaining}")
print("✅ 실습 2 완료! TokenBudget 기본 동작이 올바릅니다.")

## 실습 3 솔루션: TokenBudget 실전 활용

In [ ]:
large_code = '''
class DataProcessor:
    """데이터를 처리하는 클래스."""
    
    def __init__(self, config: dict):
        self.config = config
        self.cache = {}
    
    def process(self, data: list) -> list:
        """데이터를 처리합니다."""
        results = []
        for item in data:
            validated = self._validate(item)
            if validated:
                transformed = self._transform(validated)
                results.append(transformed)
        return results
    
    def _validate(self, item: dict) -> dict | None:
        if "id" not in item:
            return None
        return item
    
    def _transform(self, item: dict) -> dict:
        return {"id": item["id"], "value": item.get("value", 0) * 2}
''' * 3

budget = TokenBudget(max_tokens=500)
system_prompt = "당신은 코드 분석 전문가입니다. 다음 코드를 분석하세요."
user_query = "이 코드에서 버그를 찾아주세요."

budget.consume(system_prompt)
print(f"시스템 프롬프트 후 남은 토큰: {budget.remaining}")

budget.consume(user_query)
print(f"사용자 질문 후 남은 토큰: {budget.remaining}")

truncated_code = budget.truncate_to_budget(large_code, reserve=50)
print(f"코드 추가 후 남은 토큰: {budget.remaining}")

In [ ]:
# 검증
assert budget is not None
assert truncated_code is not None

budget_check = TokenBudget(max_tokens=500)
original_tokens = budget_check.count(large_code)
truncated_tokens = budget_check.count(truncated_code)

assert truncated_tokens < original_tokens
print(f"원본 코드 토큰: {original_tokens}")
print(f"절삭 후 코드 토큰: {truncated_tokens}")
print(f"절감률: {(1 - truncated_tokens/original_tokens)*100:.0f}%")
print("✅ 실습 3 완료! TokenBudget으로 프롬프트를 효율적으로 관리했습니다.")